# VT false-alarm project
Run this notebook from the extracted project folder. It calls the same tested Python package as the command line. Do not mix toy demo results with real VTaC findings.

In [ ]:
%pip install -e .

## 1. Verify setup
The following demo uses generated data. Choose a fresh output directory if it already exists.

In [ ]:
!python -m unittest discover -s tests -v
!python -m vt_alarm.cli demo --out my_demo --epochs 3 --bootstrap 100

## 2. Download the real dataset
This downloads labeled waveforms only and can take time. Leave the official split unchanged.

In [ ]:
!python -m vt_alarm.cli download --data data/vtac

## 3. Prepare and inspect
Inspect exclusions and class balance before proceeding.

In [ ]:
!python -m vt_alarm.cli prepare --data data/vtac --cache cache/vtac --seconds 30
import pandas as pd
audit = pd.read_csv("cache/vtac/audit.csv")
display(audit.groupby(["split", "status"]).size())
display(audit[audit.status == "excluded"])

## 4. Train and freeze
Validation selects settings and thresholds. Choose a fresh run folder for a documented validation-driven revision.

In [ ]:
!python -m vt_alarm.cli train --cache cache/vtac --out runs/main --epochs 40 --seeds 11 22 33

## 5. Final held-out evaluation
Run after freezing the pipeline. Do not tune against these results.

In [ ]:
!python -m vt_alarm.cli evaluate --cache cache/vtac --out runs/main --bootstrap 1000
display(pd.read_csv("runs/main/evaluation/metrics.csv"))

In [ ]:
from IPython.display import Markdown, display, Image
display(Markdown(open("runs/main/evaluation/REPORT.md").read()))
display(Image("runs/main/evaluation/curves.png"))